# Regrid HTAP3 aircraft historical and SSP245 emissions

In [1]:
sys.path.insert(0,'/glade/u/home/emmons/python/')
import numpy as np
import xarray as xr
import esmpy as ESMF
import glob
import datetime
from functions.array.chk import chk
from functions.Plot_2D import Plot_2D
from functions.Calc_Emis import Calc_Emis_T
from functions.Regridding_ESMF import Add_bounds, Regridding
from functions.io.get_sp import get_sp
import multiprocessing, psutil

## Directory and filename setup
Run for historical and SSP separately.  Then need to concatenate files for each species.
SO2 hist file has different creation date.
After files were created moved to /glade/campaign/acom/acom-weather/emmons/HTAP3/emissions/aircraft/f09/units_kg/.

In [35]:
path_air = '/glade/campaign/acom/acom-weather/emmons/HTAP3/emissions/aircraft/orig_0.5deg/'

# historical emissions dir & filenames
date_range = '200001-201412'
#file_list = glob.glob( path_air + '*-em-AIR-anthro_input4MIPs_emissions_CMIP_CEDS-2017-08-30_gn_'+date_range+'-thorfixed.nc' )
file_so2 = path_air + 'SO2-em-AIR-anthro_input4MIPs_emissions_CMIP_CEDS-2017-10-05_gn_200001-201412-thorfixed.nc'
# SSP emissions
#date_range = '201501-210012'
#file_list = glob.glob( path_air + '*-em-AIR-anthro_input4MIPs_emissions_ScenarioMIP_IAMC-MESSAGE-GLOBIOM-ssp245-1-1_gn_'+date_range+'-thorfixed.nc' )

# Destination filename format that regridded field will be saved
# "SPC" will be replaced to the real species name
dst_path = '/glade/campaign/acom/acom-weather/emmons/HTAP3/emissions/aircraft/f09/'
dst_file_format = dst_path+'HTAP3-aircraft_'+date_range+'_f09_SPC.nc'
dst_grid_file = '/glade/campaign/acom/acom-weather/emmons/emissions/grids/Gridinfo_CESM_f09.nc'

In [36]:
# set list of files to use
#species, species_files = get_sp( file_list )
species_files = file_list 

#species = ['NOx']
#species = ['BC', 'CO', 'OC']
species = ['SO2']
sectors = []
species

['SO2']

## Make grid information NetCDF file (if "lon_bnds" and "lat_bnds" are not available for FV grid input)

In [37]:
orig_grid_file = '/glade/campaign/acom/acom-weather/emmons/HTAP3/emissions/aircraft/orig_0.5deg/Gridinfo_HTAP3-AIR_c20251126.nc'
# Uncomment below if you need to create grid information file
#Add_bounds( file_list[0], orig_grid_file, creation_date=False )

## Create weight file if you don't have one already. This is extremely useful especially when you have more than two files to be processed

In [38]:
# Regridding weight file (to be created here or if you have already)
Regridding_weights_file = '/glade/campaign/acom/acom-weather/emmons/HTAP3/emissions/aircraft/orig_0.5deg/ESMF_Weight_HTAP3-AIR_grid_to_f09_Conserve_c20251126.nc'

# Uncomment below if you need to create new weight file
#ds_1 = xr.open_dataset( file_list[0] )
#rr = Regridding( ds_1.isel(time=slice(0,1)), src_grid_file=orig_grid_file, dst_grid_file=dst_grid_file, 
#                 wgt_file=Regridding_weights_file, method='Conserve', save_wgt_file=True, save_results=False,
#                 save_wgt_file_only=True, check_timings=True, creation_date=False )

## Make weight file for regridding (if not available, just create once for different species with same grid source and destination). 
### - You can just create weight file, or also you can do regridding at once. 
### - You can also save NetCDF file automatically or just can get regridded field in this python shell. 
### - You can specify sectors of your interest or pass it as is for regridding all fields at once 
### - You can check the results by check_results=True, to see whether the regridding conserves mass 
### - You can check the timings - time taken to regridding
### - You can provide mw and unit and force the tool to use them if original file has wrong mw or unit information
### - Input array can be either normal array or xarray, but xarray is preferred

In [41]:
for sp1 in species:

    print( 'Regridding: ', sp1, datetime.datetime.now() )
    print( '************************************************************************' )
    #ds_emis = xr.open_dataset( species_files[sp1] )  
    #dst_file = dst_file_format.replace( 'SPC', sp1 )
    ds_emis = xr.open_dataset( species_files[0] )  
    dst_file = dst_path+'HTAP3-aircraft_'+date_range+'_f09_SO2.nc'

    rr = Regridding( ds_emis, src_grid_file=orig_grid_file, dst_grid_file=dst_grid_file, 
                     wgt_file=Regridding_weights_file, method='Conserve', fields=sectors,
                     dst_file=dst_file, save_wgt_file=False, save_results=True, check_results=False, 
                     check_timings=True, creation_date=True, nc_file_format='NETCDF4' )


Regridding:  SO2 2025-11-26 15:48:41.822655
************************************************************************
Initialization start:  2025-11-26 15:48:41.890310
Initialization end:  2025-11-26 15:48:41.893599
Time spent: 00 minutes and 00 seconds
Grid/Field setup start:  2025-11-26 15:48:41.893632
Grid/Field setup end:  2025-11-26 15:48:41.897624
Time spent: 00 minutes and 00 seconds
Read regridding weight start:  2025-11-26 15:48:41.897658
Read regridding weight end:  2025-11-26 15:48:42.731338
Time spent: 00 minutes and 00 seconds
Saving NetCDF file / regridding start:  2025-11-26 15:48:42.731640
Regridding start:  2025-11-26 15:48:42.763101
Regridding end:  2025-11-26 16:04:30.263359
Time spent: 15 minutes and 47 seconds
Saving NetCDF file / regridding end:  2025-11-26 16:04:30.301090
Time spent: 15 minutes and 47 seconds
